In [1]:
import numpy as np
import pandas as pd
%run catalogo.ipynb

(909, 9)
       articolo     brand        fornitore modalita_acquisto categoria    colore stagione  prezzo_listino  costo_unitario
0  AGO-CAM-0001  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     verde     PE23           109.0           40.47
1  AGO-CAM-0002  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           119.0           47.71
2  AGO-CAM-0003  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  fantasia     PE23           129.0           48.24
3  AGO-CAM-0004  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23            79.0           30.54
4  AGO-CAM-0005  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia    bianco     PE23           139.0           52.70
5  AGO-CAM-0006  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           129.0           49.90
6  AGO-CRA-0007  Ago&Filo  Ago&Filo S.r.l.             fermo  cravatta     rosso     PE23            79.0           31.77
7  AGO-CRA-0008

In [2]:
SCALE = {
    "lettere": (["XS", "S", "M", "L", "XL", "XXL"],
                [0.03, 0.15, 0.30, 0.30, 0.15, 0.07]),
    "numeri": (["44", "46", "48", "50", "52", "54", "56"],
                [0.03, 0.10, 0.22, 0.27, 0.22, 0.11, 0.05]),
    "scarpe": (["41", "42", "43", "44", "45", "46"],
                [0.12, 0.25, 0.26, 0.20, 0.12, 0.05]),
    "calzini": (["39-42", "43-46"], 
                [0.45, 0.55]),
    "unica": (["TU"], [1.0]),
}

In [3]:
for nome, (taglie, pesi) in SCALE.items():
    assert len(taglie) == len(pesi), f"{nome}: taglie e pesi di lunghezza diversa"
    assert abs(sum(pesi) -1) < 1e-9, f"{nome}: i pesi sommano a {sum(pesi)}"

In [4]:
SCALA_PER_CATEGORIA = {
    "t-shirt": "lettere", "camicia": "lettere", "maglia": "lettere", "felpa": "lettere",
    "capospalla": "lettere", "costume": "lettere", "pantalone": "numeri",
    "jeans": "numeri", "giacca": "numeri", "abito": "numeri",
    "scarpe eleganti": "scarpe", "scarpe casual": "scarpe", 
    "calzini": "calzini", "cravatta": "unica", "profumo": "unica",
}

In [5]:
PEZZI_PER_ARTICOLO = {
     "t-shirt": (26, 42), "camicia": (20, 36), "maglia": (16, 30), "felpa": (16,30),
    "capospalla": (10, 20), "costume": (20, 36), "pantalone": (20, 36),
    "jeans": (26, 42), "giacca": (10, 20), "abito": (8, 16),
    "scarpe eleganti": (12, 24), "scarpe casual": (16, 26), 
    "calzini": (30, 52), "cravatta": (10, 20), "profumo": (10, 20),
}

In [6]:
assert set(SCALA_PER_CATEGORIA) == set(PREZZI) == set(PEZZI_PER_ARTICOLO), "categorie non allineate"

In [7]:
def date_campionario(stagione):
    """Finestre di ordine e arrivo: PE ordinata in autunno dell'anno prima, AI a fine inverno."""
    anno = 2000 + int(stagione[2:])
    if stagione.startswith("PE"):
        ordine = (f"{anno - 1}-09-15", f"{anno - 1}-10-31")
        arrivo = (f"{anno}-02-01", f"{anno}-03-15")
    else:
        ordine = (f"{anno}-02-15", f"{anno}-03-31")
        arrivo = (f"{anno}-08-01", f"{anno}-09-15")
    return ordine, arrivo

In [8]:
def data_casuale(inizio, fine):
    giorni = pd.date_range(inizio, fine)
    return giorni[rng.integers(len(giorni))]

In [9]:
def calendario_consegne(catalogo):
    """Un ordine per brand e stagione; la merce arriva in 1 o 2 consegne."""
    consegne = {}
    for brand, stagione in catalogo[["brand", "stagione"]].drop_duplicates().itertuples(index=False):
        (o_ini, o_fin), (a_ini, a_fin) = date_campionario(stagione)
        data_ordine = data_casuale(o_ini, o_fin)
        prima = data_casuale(a_ini, a_fin)
        arrivi = [prima]
        if rng.random() < 0.5:
            arrivi.append(prima + pd.Timedelta(days=int(rng.integers(14, 28, endpoint=True))))
        consegne[(brand, stagione)] = (data_ordine, arrivi)
    return consegne

In [10]:
def crea_ordini(catalogo, fattori=None):
    fattori = fattori or {}
    fill = BRAND.set_index("brand")["fill_rate"]
    consegne = calendario_consegne(catalogo)
    righe = []
    for _, a in catalogo.iterrows():
        taglie, pesi = SCALE[SCALA_PER_CATEGORIA[a["categoria"]]]
        totale = rng.integers(*PEZZI_PER_ARTICOLO[a["categoria"]], endpoint=True)
        f = fattori.get((a["brand"], a["categoria"]), 1.0)
        totale = max(1, int(round(totale * f)))
        quantita = rng.multinomial(totale, pesi)
        data_ordine, arrivi = consegne[(a["brand"], a["stagione"])]
        data_arrivo = arrivi[rng.integers(len(arrivi))]
        for taglia, qta in zip(taglie, quantita):
            if qta == 0:
                continue
            ricevuta = qta
            if fill[a["brand"]] < 1 and rng.random() < 0.4:
                ricevuta = int(round(qta * rng.uniform(0.3, 0.7)))
            righe.append({
                "articolo": a["articolo"], "taglia": taglia,
                "brand": a["brand"], "fornitore": a["fornitore"],
                "modalita_acquisto": a["modalita_acquisto"],
                "categoria": a["categoria"], "colore": a["colore"],
                "stagione": a["stagione"], "tipo_ordine": "campionario",
                "data_ordine": data_ordine.date(), "data_arrivo": data_arrivo.date(),
                "qta_ordinata": int(qta), "qta_ricevuta": int(ricevuta),
                "costo_unitario": a["costo_unitario"], "prezzo_listino": a["prezzo_listino"],
            })
    return pd.DataFrame(righe)

In [11]:
ordini = crea_ordini(cat)
print(ordini.shape)
print(ordini.head(10).to_string())
print("\nPezzi ricevuti / ordinati per brand:")
tot = ordini.groupby("brand")[["qta_ordinata", "qta_ricevuta"]].sum()
tot["fill_rate"] = (tot["qta_ricevuta"] / tot["qta_ordinata"]).round(2)
print(tot)
print("\nPezzi ricevuti per stagione:")
print(ordini.groupby("stagione")["qta_ricevuta"].sum())
print("\nArrivo prima dell'ordine:", (ordini["data_arrivo"] < ordini["data_ordine"]).sum())
print("Chiave articolo+taglia duplicata:", ordini.duplicated(["articolo", "taglia"]).sum())
print("\nDate d'arrivo diverse per brand e stagione:")
print(ordini.groupby(["stagione", "brand"])["data_arrivo"].nunique().unstack(0))

(4229, 15)
       articolo taglia     brand        fornitore modalita_acquisto categoria colore stagione  tipo_ordine data_ordine data_arrivo  qta_ordinata  qta_ricevuta  costo_unitario  prezzo_listino
0  AGO-CAM-0001     XS  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  verde     PE23  campionario  2022-10-05  2023-02-13             1             1           40.47           109.0
1  AGO-CAM-0001      S  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  verde     PE23  campionario  2022-10-05  2023-02-13             1             1           40.47           109.0
2  AGO-CAM-0001      M  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  verde     PE23  campionario  2022-10-05  2023-02-13             8             8           40.47           109.0
3  AGO-CAM-0001      L  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  verde     PE23  campionario  2022-10-05  2023-02-13             8             8           40.47           109.0
4  AGO-CAM-0001     XL  Ago&Filo  